In [8]:
import pygad
from scipy.stats import norm
import matplotlib.pyplot as plt
import json
from fitness import *
import numpy as np

In [9]:
with open("datasets/medium_dataset_1.json", "r") as file:
    data = json.load(file)

# Step 2: Extract information
dataset_id = data["dataset_id"]
num_papers = data["num_papers"]
num_reviewers = data["num_reviewers"]
reviewer_capacity = data["reviewer_capacity"]
min_reviews_per_paper = data["min_reviews_per_paper"]
max_reviews_per_paper = data["max_reviews_per_paper"]
preferences = np.array(data["preferences"])
friendship = np.array(data["friendships"])
authorship = np.array(data["authorship"])

In [10]:
def check(ga_instance, children):
    # Repair children
    for idx, child in enumerate(children):
        matr = np.reshape(child, (num_reviewers, num_papers))
        if not is_valid_assignment(matr):
            children[idx] = repair_solution(matr).reshape(-1)
    return children

def check_initial(ga_instance):
    # Repair initial population
    for solution_idx, solution in enumerate(ga_instance.population):
        assignment = solution.reshape((num_reviewers, num_papers))
        if not is_valid_assignment(assignment):
            ga_instance.population[solution_idx] = repair_solution(assignment).reshape(-1)

# Validity checks (constraints)
def is_valid_assignment(assignment):
    # Check paper reviews (min and max reviews per paper)  - Skipped because with our correction algorithm, we are not fixing this constraint 
    # (cannot satisfy all constraints with simple correction ... or?)
    #
    # paper_counts = np.sum(assignment, axis=0)
    # if any(paper_counts < min_reviews_per_paper) or any(paper_counts > max_reviews_per_paper):
    #     return False
    
    # Check reviewer capacities
    reviewer_counts = np.sum(assignment, axis=1)
    if any(reviewer_counts > reviewer_capacity):
        return False
    
    # Check authorship constraints (a reviewer cannot review their own paper)
    if np.any(assignment & authorship):
        return False

    # Check friendship constraints (no two friends should review the same paper)
    for i in range(num_reviewers):
        for j in range(num_reviewers):
            if friendship[i][j] == 1:
                for k in range(num_papers):
                    if assignment[i][k] == 1 and assignment[j][k] == 1:
                        return False

    return True

# Repair function to correct an invalid solution
def repair_solution(assignment):
    # Ensure reviewer capacities are respected
    reviewer_counts = np.sum(assignment, axis=1)
    for i in range(num_reviewers):
        if reviewer_counts[i] > reviewer_capacity:
            for j in range(num_papers):
                if assignment[i][j] == 1:
                    assignment[i][j] = 0
                    reviewer_counts = np.sum(assignment, axis=1)
                    if reviewer_counts[i] <= reviewer_capacity:
                        break
    
    # Ensure friendship constraints are respected
    for i in range(num_reviewers):
        for j in range(num_reviewers):
            if friendship[i][j] == 1:
                for k in range(num_papers):
                    if assignment[i][k] == 1 and assignment[j][k] == 1:
                        assignment[i][k] = 0  # Remove one of them
                        break
    
    # Ensure authorship constraints are respected
    assignment *= (1-authorship)
    
    return assignment

# Fitness function
def fitness_func(ga_instance, solution, solution_idx):
    assignment = solution.reshape(num_reviewers, num_papers)
    
    # Penalty weights
    rlpp = 10
    # plpr = 10
    # fc = 20
    # ac = 50

    # Satisfaction score
    satisfaction = np.sum(assignment * preferences)
    
    # Constraints penalties
    penalties = 0
    
    # Review limit per paper
    for j in range(num_papers):
        if np.sum(assignment[:, j]) < min_reviews_per_paper or np.sum(assignment[:, j]) > max_reviews_per_paper:
            penalties += rlpp
    
    # - Skipped since only Review limit per paper constraint can be violated with our repair algorithm
    # Paper limit per reviewer
    # for i in range(num_reviewers):
    #     if np.sum(assignment[i, :]) > reviewer_capacity:
    #         penalties += plpr
    #
    # # Friendship constraint
    # for j in range(num_papers):
    #     reviewers = np.where(assignment[:, j] == 1)[0]
    #     for r1 in reviewers:
    #         for r2 in reviewers:
    #             if friendship[r1, r2] == 1:
    #                 penalties += fc
    
    # # Authorship constraint
    # if np.any(assignment * authorship):
    #     penalties += ac
    
    return satisfaction - penalties

binary_gene_space = [0,1]
# Define the genetic algorithm using the custom GA class


In [11]:
num_iter = 20
avg = 0
seed = 0

for i in range(num_iter):
    seed += 1
    ga_instance = pygad.GA(
        num_generations=100,
        num_parents_mating=20,
        fitness_func=fitness_func,
        sol_per_pop=500,
        num_genes=num_reviewers * num_papers,
        gene_type=int,
        gene_space=binary_gene_space,
        mutation_type="swap", 
        crossover_type="single_point", 
        mutation_probability=0.2,  # Dynamic mutation
        keep_elitism=10,  # Preserve the top 10 solutions
        on_start=check_initial,
        on_mutation=check,
        on_crossover=check,
        random_seed=seed
    )
    ga_instance.run()

    best_solution, best_solution_fitness, _ = ga_instance.best_solution()
    ff = eval_ff(best_solution, preferences, num_reviewers, num_papers, max_reviews_per_paper, min_reviews_per_paper, reviewer_capacity, authorship, friendship)
    print(ff)
    avg += ff

print(avg/20)

/home/matjaz/.conda/envs/IS/lib/python3.13/site-packages/pygad/pygad.py:1139: UserWarning: The 'delay_after_gen' parameter is deprecated starting from PyGAD 3.3.0. To delay or pause the evolution after each generation, assign a callback function/method to the 'on_generation' parameter to adds some time delay.
  warnings.warn("The 'delay_after_gen' parameter is deprecated starting from PyGAD 3.3.0. To delay or pause the evolution after each generation, assign a callback function/method to the 'on_generation' parameter to adds some time delay.")


Num papers constraint fail. Column:  1
Num papers constraint fail. Column:  3
69
Num papers constraint fail. Column:  0
Num papers constraint fail. Column:  1
Num papers constraint fail. Column:  5
56
Num papers constraint fail. Column:  1
86
Num papers constraint fail. Column:  2
Num papers constraint fail. Column:  5
Num papers constraint fail. Column:  8
47
Num papers constraint fail. Column:  1
Num papers constraint fail. Column:  2
Num papers constraint fail. Column:  3
Num papers constraint fail. Column:  9
42
Num papers constraint fail. Column:  1
Num papers constraint fail. Column:  5
68
Num papers constraint fail. Column:  0
Num papers constraint fail. Column:  1
56
Num papers constraint fail. Column:  0
Num papers constraint fail. Column:  1
Num papers constraint fail. Column:  2
55
Num papers constraint fail. Column:  0
Num papers constraint fail. Column:  5
61
Num papers constraint fail. Column:  0
Num papers constraint fail. Column:  1
Num papers constraint fail. Column:  